# Part 2: Bayesian Probability – IMDb Movie Reviews
We use **Bayes' Theorem** to find how likely a review is positive when it contains a keyword:

$$P(\text{Positive} \mid kw) = \frac{P(kw \mid \text{Positive}) \times P(\text{Positive})}{P(kw)}$$

We chose to compute **P(Positive | keyword)** only, as the brief requires one direction.
Basic Python only: `csv` and `re` from the standard library, no ML libraries.

In [3]:
import glob
from IPython.display import Markdown, display
import bayes

CSV_PATH = glob.glob("*.csv")[0]   # the IMDb CSV in this folder
print("Using:", CSV_PATH)
reviews = bayes.load_reviews(CSV_PATH)

Using: IMDB Dataset.CSV
Loaded 50000 reviews (0 rows skipped).


## 1. How we clean the text (tokenization)
- Lowercase everything, so "Great" and "great" are the same word.
- Remove HTML tags like `<br />`, which appear in many IMDb reviews.
- Keep only words (letters, plus forms like "don't"), so "great!" becomes "great".
- Count each word **once per review**: we ask "does this review contain the word?", not "how many times?".

In [4]:
print(sorted(bayes.tokenize("This movie was GREAT!<br /><br />I don't regret it, great acting.")))

['acting', "don't", 'great', 'i', 'it', 'movie', 'regret', 'this', 'was']


## 2. Keyword choices and why
| Keyword | Expected | Why we chose it |
|---|---|---|
| excellent | Positive | A strong, direct word of praise that is rarely used sarcastically. |
| wonderful | Positive | Expresses enjoyment and emotion, common in reviews of liked films. |
| masterpiece | Positive | The highest praise for a film; people use it when they loved it. |
| awful | Negative | A strong, direct word of dislike. |
| waste | Negative | Used in "waste of time / money", a classic complaint. |
| boring | Negative | The most common reason people give for disliking a film. |

In [5]:
KEYWORDS = ["excellent", "wonderful", "masterpiece", "awful", "waste", "boring"]
results = [bayes.keyword_probabilities(reviews, kw) for kw in KEYWORDS]

## 3. The four probabilities for each keyword

In [6]:
for result in results:
    display(Markdown(bayes.markdown_table(result)))

**Keyword: excellent** (in 3552 reviews)

| Probability | Value |
|---|---|
| Prior P(Positive) | 0.5000 |
| Likelihood P(excellent \| Positive) | 0.1147 |
| Marginal P(excellent) | 0.0710 |
| Posterior P(Positive \| excellent) | 0.8074 |


**Keyword: wonderful** (in 2778 reviews)

| Probability | Value |
|---|---|
| Prior P(Positive) | 0.5000 |
| Likelihood P(wonderful \| Positive) | 0.0902 |
| Marginal P(wonderful) | 0.0556 |
| Posterior P(Positive \| wonderful) | 0.8121 |


**Keyword: masterpiece** (in 1204 reviews)

| Probability | Value |
|---|---|
| Prior P(Positive) | 0.5000 |
| Likelihood P(masterpiece \| Positive) | 0.0351 |
| Marginal P(masterpiece) | 0.0241 |
| Posterior P(Positive \| masterpiece) | 0.7284 |


**Keyword: awful** (in 2882 reviews)

| Probability | Value |
|---|---|
| Prior P(Positive) | 0.5000 |
| Likelihood P(awful \| Positive) | 0.0113 |
| Marginal P(awful) | 0.0576 |
| Posterior P(Positive \| awful) | 0.0982 |


**Keyword: waste** (in 2533 reviews)

| Probability | Value |
|---|---|
| Prior P(Positive) | 0.5000 |
| Likelihood P(waste \| Positive) | 0.0070 |
| Marginal P(waste) | 0.0507 |
| Posterior P(Positive \| waste) | 0.0687 |


**Keyword: boring** (in 3052 reviews)

| Probability | Value |
|---|---|
| Prior P(Positive) | 0.5000 |
| Likelihood P(boring \| Positive) | 0.0237 |
| Marginal P(boring) | 0.0610 |
| Posterior P(Positive \| boring) | 0.1940 |


## 4. Checking our Bayes implementation
Bayes' Theorem must give the same answer as counting directly (positive reviews with the keyword ÷ all reviews with the keyword).

In [7]:
for result in results:
    direct = bayes.direct_check(reviews, result["keyword"])
    same = (direct is None and result["posterior"] is None) or abs(direct - result["posterior"]) < 1e-12
    print(f"{result['keyword']:<12} Bayes = {result['posterior']}   direct = {direct}   match = {same}")

excellent    Bayes = 0.8074324324324323   direct = 0.8074324324324325   match = True
wonderful    Bayes = 0.8120950323974082   direct = 0.8120950323974082   match = True
masterpiece  Bayes = 0.728405315614618   direct = 0.728405315614618   match = True
awful        Bayes = 0.09819569743233866   direct = 0.09819569743233865   match = True
waste        Bayes = 0.06869324911172524   direct = 0.06869324911172522   match = True
boring       Bayes = 0.19397116644823068   direct = 0.19397116644823068   match = True


## 5. Absent keywords
If a keyword never appears, P(keyword) = 0 and Bayes' Theorem would divide by zero. Our code detects this and reports it instead of crashing:

In [8]:
display(Markdown(bayes.markdown_table(bayes.keyword_probabilities(reviews, "zzzznotaword"))))

**zzzznotaword**: not found in any review, so P(keyword) = 0 and the posterior is undefined.

## 6. Interpretation

In [9]:
for result in results:
    print("-", bayes.interpret(result))

- A review containing 'excellent' is 80.7% likely to be positive, versus 50.0% for a random review, so it is more likely to be positive.
- A review containing 'wonderful' is 81.2% likely to be positive, versus 50.0% for a random review, so it is more likely to be positive.
- A review containing 'masterpiece' is 72.8% likely to be positive, versus 50.0% for a random review, so it is more likely to be positive.
- A review containing 'awful' is 9.8% likely to be positive, versus 50.0% for a random review, so it is less likely to be positive (leans negative).
- A review containing 'waste' is 6.9% likely to be positive, versus 50.0% for a random review, so it is less likely to be positive (leans negative).
- A review containing 'boring' is 19.4% likely to be positive, versus 50.0% for a random review, so it is less likely to be positive (leans negative).


## 7. Conclusion

The dataset has 50,000 reviews, half positive and half negative, so the prior P(Positive) is 0.5. This means any keyword that pushes the posterior above 50% is a sign of a positive review, and below 50% a sign of a negative one.

Positive keywords: "excellent" appears in 3,552 reviews, and 80.7% of them are positive. "wonderful" gives [...]% and "masterpiece" gives [...]%. All three are clearly above 50%, so our choices were good positive signals.

Negative keywords: "awful" gives only [...]% positive, "waste" [...]%, and "boring" [...]%. These are well below 50%, so they are strong signs of a negative review.

Strongest signals: the best positive keyword was [...] and the strongest negative one was [...].

Why no keyword is 100%: some people use positive words in negative reviews ("excellent cast but a bad film") or say "not boring" in a good review. Bayes only looks at whether the word appears, not at its context, so a single keyword is a strong hint but never a guarantee.

